In [21]:
!pip install langchain langchain-google-genai sentence-transformers faiss-cpu


[notice] A new release of pip is available: 23.0.1 -> 26.1
[notice] To update, run: C:\Users\ASUS\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


In [22]:
import json
import os
import requests
from typing import List, Dict
from langchain_openai import ChatOpenAI
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

In [23]:
os.environ["OPENAI_API_KEY"] = "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"
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
llm = ChatOpenAI(model="mistralai/ministral-8b-2512", temperature=0.1)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [24]:
class ReviewerDataLoader:
    
    def __init__(self, json_file_path: str):
        self.json_file_path = json_file_path
        self.reviewer_data = []
        
    def load_data(self, limit: int = None):
        print(f"📂 Loading reviewer data from {self.json_file_path}...")
        
        try:
            with open(self.json_file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract PR reviewer data
            pr_reviewer_data = data.get('pr_reviewer_data', [])
            
            # Apply limit only if specified
            if limit:
                self.reviewer_data = pr_reviewer_data[:limit]
                print(f"✅ Loaded {len(self.reviewer_data)} PRs with reviews (limited from {len(pr_reviewer_data)} total)")
            else:
                self.reviewer_data = pr_reviewer_data
                print(f"✅ Loaded ALL {len(self.reviewer_data)} PRs with reviews")
            
            return True
            
        except Exception as e:
            print(f"❌ Error loading data: {e}")
            return False
    
    def get_pr_summary(self, pr: Dict) -> str:
        author = pr.get('author', {}).get('username', 'Unknown') if pr.get('author') else 'Unknown'
        
        # Get repository and project info
        repo_name = pr.get('repo_name', 'Unknown Repository')
        
        # Get reviews info
        reviews = pr.get('reviews', [])
        review_summary = []
        if reviews:
            for review in reviews[:3]:  # Show first 3 reviews
                reviewer = review.get('reviewer_username', 'Unknown')
                state = review.get('state', 'unknown')
                review_summary.append(f"  - {reviewer}: {state}")
        
        # Get review comments
        review_comments = pr.get('review_comments', [])
        
        # Format created and updated dates
        created_at = pr.get('created_at', 'Unknown')
        updated_at = pr.get('updated_at', 'Unknown')
        
        summary = f"""
REPOSITORY: {repo_name}
PR #{pr.get('pr_number', 'N/A')}: {pr.get('title', 'No title')}
Author: {author}
State: {pr.get('state', 'unknown')}
Is Draft: {pr.get('is_draft', False)}
Created: {created_at}
Updated: {updated_at}

DESCRIPTION:
{pr.get('description', 'No description')}

CHANGE STATISTICS:
- Files Changed: {pr.get('changed_files_count', 0)}
- Additions: +{pr.get('additions', 0)} lines
- Deletions: -{pr.get('deletions', 0)} lines

LABELS: {', '.join(pr.get('labels', [])) if pr.get('labels') else 'None'}

REVIEWS ({len(reviews)} total):
{chr(10).join(review_summary) if review_summary else 'No reviews yet'}

REVIEW COMMENTS: {len(review_comments)} comments
"""
        return summary.strip()

# Load TEST reviewer data (kept for non-LLM/test flow)
reviewer_data_file = "../PR data for Reviewers/expressjs_express_reviewer_data_test.json"

print("🚀 Loading reviewer TEST data...")
reviewer_loader = ReviewerDataLoader(reviewer_data_file)
success = reviewer_loader.load_data()

if success:
    print(f"🎉 Successfully loaded TEST reviewer data!")
else:
    print("❌ Failed to load TEST reviewer data")

# Load TRAIN reviewer data (used only for FAISS to avoid data leakage)
train_reviewer_data_file = reviewer_data_file.replace("_test.json", "_train.json")
train_reviewer_loader = ReviewerDataLoader(train_reviewer_data_file)

if os.path.exists(train_reviewer_data_file):
    print("🚀 Loading reviewer TRAIN data for FAISS...")
    train_success = train_reviewer_loader.load_data()
    if train_success:
        print(f"🎉 Successfully loaded TRAIN reviewer data for FAISS!")
    else:
        print("❌ Failed to load TRAIN reviewer data for FAISS")
else:
    print(f"❌ TRAIN data file not found: {train_reviewer_data_file}")
    print("⚠️ FAISS creation will be skipped until train data is available")

🚀 Loading reviewer TEST data...
📂 Loading reviewer data from ../PR data for Reviewers/expressjs_express_reviewer_data_test.json...
✅ Loaded ALL 52 PRs with reviews
🎉 Successfully loaded TEST reviewer data!
🚀 Loading reviewer TRAIN data for FAISS...
📂 Loading reviewer data from ../PR data for Reviewers/expressjs_express_reviewer_data_train.json...
✅ Loaded ALL 121 PRs with reviews
🎉 Successfully loaded TRAIN reviewer data for FAISS!


In [25]:
class ReviewerVectorStore:
    
    def __init__(self, embeddings_model):
        self.embeddings = embeddings_model
        self.vector_store = None
        self.documents = []
    
    def create_embeddings(self, reviewer_data: List[Dict]):
        """Convert reviewer PR data to vector embeddings"""
        print("🔄 Creating vector embeddings for reviewer data...")
        
        documents = []
        
        # Convert each PR with reviews to a document
        for pr in reviewer_data:
            # Create text content with reviewer information
            author = pr.get('author', {}).get('username', 'Unknown') if pr.get('author') else 'Unknown'
            repo_name = pr.get('repo_name', 'Unknown Repository')
            
            # Get reviewer information
            reviews = pr.get('reviews', [])
            reviewers = []
            review_states = []
            review_bodies = []
            
            for review in reviews:
                reviewers.append(review.get('reviewer_username', 'Unknown'))
                review_states.append(review.get('state', 'unknown'))
                if review.get('body'):
                    review_bodies.append(review.get('body', '')[:200])  # First 200 chars
            
            # Get review comments
            review_comments = pr.get('review_comments', [])
            comment_authors = []
            for comment in review_comments[:5]:  # First 5 comments
                if comment.get('user', {}).get('login'):
                    comment_authors.append(comment.get('user', {}).get('login'))
            
            content = f"""
Repository: {repo_name}
PR #{pr.get('pr_number')}: {pr.get('title', '')}
Author: {author}
Description: {pr.get('description', '')[:300]}...
Labels: {', '.join(pr.get('labels', []))}
State: {pr.get('state', 'unknown')}
Is Draft: {pr.get('is_draft', False)}
Files: {pr.get('changed_files_count', 0)} changed
Changes: +{pr.get('additions', 0)} -{pr.get('deletions', 0)}
Reviewers: {', '.join(reviewers)}
Review States: {', '.join(review_states)}
Review Comments Authors: {', '.join(comment_authors)}
Sample Review Content: {' | '.join(review_bodies[:2])}
"""
            
            # Create metadata
            metadata = {
                'pr_number': pr.get('pr_number'),
                'author': author,
                'state': pr.get('state'),
                'title': pr.get('title', ''),
                'repo_name': repo_name,
                'reviewers': reviewers,
                'review_count': len(reviews),
                'comment_count': len(review_comments)
            }
            
            # Create document
            doc = Document(page_content=content.strip(), metadata=metadata)
            documents.append(doc)
        
        # Create vector store
        self.vector_store = FAISS.from_documents(documents, self.embeddings)
        self.documents = documents
        
        print(f"✅ Created {len(documents)} vector embeddings for reviewer data")
        return self.vector_store
    
    def find_similar_reviewed_prs(self, query: str, k: int = 5) -> List[Document]:
        """Find similar PRs that have been reviewed"""
        if not self.vector_store:
            print("❌ Vector store not created yet!")
            return []
        
        # Perform similarity search
        similar_docs = self.vector_store.similarity_search(query, k=k)
        return similar_docs

# Create vector store for reviewer data
# IMPORTANT: build FAISS from TRAIN data only to avoid test leakage.
reviewer_vector_store = ReviewerVectorStore(embeddings)
if 'train_reviewer_loader' in globals() and train_reviewer_loader.reviewer_data:
    reviewer_vector_store.create_embeddings(train_reviewer_loader.reviewer_data)
    print("🎯 Reviewer vector store ready for similarity search (TRAIN data)!")
else:
    print("❌ No TRAIN reviewer data available for embedding")

🔄 Creating vector embeddings for reviewer data...


c:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


✅ Created 121 vector embeddings for reviewer data
🎯 Reviewer vector store ready for similarity search (TRAIN data)!


In [26]:
import glob
from typing import List, Dict, Tuple

class SmartReviewerAssigner:
    """Enhanced system that can assign the best reviewers to PRs based on reviewer profiles and past review patterns"""
    
    def __init__(self, llm_model, vector_store, profiles_directory: str = "Reviewers Profiles"):
        self.llm = llm_model
        self.vector_store = vector_store
        self.profiles_directory = profiles_directory
        self.reviewer_profiles = {}
        self.load_reviewer_profiles()
        
        # Prompt for analyzing PR requirements
        self.analysis_prompt = PromptTemplate(
            input_variables=["pr_content", "code_changes"],
            template="""You are a strict PR requirement extraction engine for reviewer assignment.

Goal: infer the minimum set of reviewer requirements from the PR content and code changes. Be precise, conservative, and evidence-based.

Rules:
- Use only facts supported by the provided PR content and code changes.
- Do not mention technologies, frameworks, or domains unless there is clear evidence.
- Prefer fewer, higher-confidence requirements over broad guesses.
- If something is not explicit, omit it or use an empty array.
- Return JSON only. No markdown, no explanation, no code fences.

PR CONTENT:
{pr_content}

CODE CHANGES:
{code_changes}

Extract these fields:
- technical_skills_needed: concrete technologies, libraries, languages, or APIs required to review this PR
- review_expertise_areas_needed: review domains such as backend, testing, security, performance, docs, release, architecture
- complexity_level: Low, Medium, or High based on scope, risk, and implementation depth
- review_focus_areas: the exact aspects a reviewer should inspect
- primary_language: the main implementation language, or Unknown if not clear
- frameworks_involved: only frameworks/libraries clearly present in the PR
- review_type_needed: the single best review type from Code Quality, Security, Performance, Documentation, Architecture, or Testing

Scoring mindset for extraction:
- Prefer explicit names and exact terms from the PR.
- Treat version bumps, dependency changes, and config-only changes as high-signal only when the changed package or config is identifiable.
- If the PR is a dependency upgrade, include the upgraded package name and the ecosystem it belongs to.
- If the PR is documentation-only, keep technical skills minimal and set review_type_needed to Documentation.
- If the PR is test-only, prioritize Testing and the runtime language/framework involved.

Return exactly this JSON shape:
{{
  "technical_skills_needed": ["skill1", "skill2"],
  "review_expertise_areas_needed": ["area1", "area2"],
  "complexity_level": "Low|Medium|High",
  "review_focus_areas": ["focus1", "focus2"],
  "primary_language": "language-or-Unknown",
  "frameworks_involved": ["framework1", "framework2"],
  "review_type_needed": "Code Quality|Security|Performance|Documentation|Architecture|Testing"
}}

Before responding, verify:
1. Every array item is supported by evidence in the input.
2. No field contains prose, bullets, or markdown.
3. The output is valid JSON with double-quoted keys and string values.
4. If evidence is weak, reduce the number of returned items rather than inventing them.

JSON:"""
        )
        
        # Prompt for matching reviewers
        self.matching_prompt = PromptTemplate(
            input_variables=["pr_requirements", "reviewer_profiles", "similar_reviews"],
            template="""You are a deterministic reviewer ranking engine.

Task: rank the best 3 reviewers for this PR using the provided requirements, reviewer profiles, and similar past reviews.

Hard rules:
- Use only reviewers that appear in AVAILABLE REVIEWERS.
- Base every claim on the provided profile data or similar review evidence.
- Treat frequency counts as the strongest signal of practical expertise.
- Prefer direct matches over general seniority.
- Penalize reviewers with weak or missing evidence for the required skills.
- Return JSON only. No markdown, no explanation, no code fences.

PR REQUIREMENTS:
{pr_requirements}

AVAILABLE REVIEWERS:
{reviewer_profiles}

SIMILAR PAST REVIEWS:
{similar_reviews}

Scoring rubric:
- skill_frequency_match: 0.00 to 1.00, weight 0.45. Measures direct overlap between required skills/frameworks and reviewer frequency evidence.
- domain_alignment: 0.00 to 1.00, weight 0.20. Measures fit to the needed review type and expertise areas.
- similar_review_relevance: 0.00 to 1.00, weight 0.15. Measures match with retrieved similar PRs and review patterns.
- review_activity_consistency: 0.00 to 1.00, weight 0.10. Measures reviewer reliability from total reviews/comments/PRs reviewed.
- complexity_fit: 0.00 to 1.00, weight 0.10. Measures whether reviewer experience matches PR complexity.

Decision guidance:
- Rank by final weighted score, highest first.
- Use conservative scores when evidence is partial.
- If two reviewers are close, prefer the one with stronger frequency evidence and more similar-review matches.
- Make the reasoning short, factual, and specific.
- Provide exactly 3 reviewers.

Return exactly this JSON shape:
{{
  "recommended_reviewers": [
    {{
      "reviewer_name": "name",
      "match_score": 0.0,
      "reasoning": "short evidence-based explanation",
      "strengths_alignment": ["specific strength 1", "specific strength 2"],
      "review_experience": "short factual summary",
      "potential_concerns": "specific gap or None identified",
      "score_breakdown": {{
        "skill_frequency_match": 0.0,
        "domain_alignment": 0.0,
        "similar_review_relevance": 0.0,
        "review_activity_consistency": 0.0,
        "complexity_fit": 0.0
      }},
      "evidence": ["evidence 1", "evidence 2"]
    }}
  ],
  "assignment_confidence": "High|Medium|Low",
  "assignment_reasoning": "one short sentence explaining the top choice",
  "ranking_notes": ["note 1", "note 2"]
}}

Validation rules:
1. match_score and all score_breakdown values must be between 0 and 1.
2. The list must be sorted by match_score descending.
3. strengths_alignment and evidence must contain only concrete evidence phrases.
4. If the data is weak, lower the scores rather than inflating confidence.

JSON:"""
        )
        
        self.analysis_chain = LLMChain(llm=self.llm, prompt=self.analysis_prompt)
        self.matching_chain = LLMChain(llm=self.llm, prompt=self.matching_prompt)
    
    def load_reviewer_profiles(self):
        """Load all reviewer profiles from JSON files"""
        profile_files = glob.glob(f"{self.profiles_directory}/*.json")
        
        print(f"🔍 Loading reviewer profiles from {self.profiles_directory}/...")
        
        for file_path in profile_files:
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    profile_data = json.load(f)
                
                reviewer_name = profile_data.get('reviewer_name')
                if reviewer_name:
                    self.reviewer_profiles[reviewer_name] = profile_data
                
            except Exception as e:
                print(f"❌ Error loading profile {file_path}: {e}")
        
        print(f"✅ Loaded {len(self.reviewer_profiles)} reviewer profiles")
        print(f"👥 Available reviewers: {', '.join(self.reviewer_profiles.keys())}")
    
    def display_reviewer_details(self, reviewer_name: str):
        """Display detailed information about a specific reviewer with frequency data"""
        if reviewer_name not in self.reviewer_profiles:
            print(f"❌ Reviewer '{reviewer_name}' not found")
            return
        
        profile_data = self.reviewer_profiles[reviewer_name]
        stats = profile_data.get('stats', {})
        profile = profile_data.get('profile', {})
        
        print(f"\n👨‍💻 REVIEWER PROFILE: {reviewer_name}")
        print("=" * 50)
        print(f"Experience Level: {profile.get('experience_level', 'Unknown')}")
        print(f"Languages: {', '.join(profile.get('programming_languages', []))}")
        print(f"Primary Skills: {', '.join(profile.get('primary_skills', []))}")
        print(f"Total Reviews: {stats.get('total_reviews', 0)}")
        print(f"Total Comments: {stats.get('total_comments', 0)}")
        print(f"Total PRs Reviewed: {stats.get('total_prs', 0)}")
        
        # Show top skills by frequency - extract from JavaScript skill matrix
        if 'javascript_skill_matrix' in profile:
            print("\nTop Skills by Experience:")
            js_matrix = profile['javascript_skill_matrix']
            skill_freq_pairs = []
            
            # Extract skills and frequencies from the matrix
            for category, skills in js_matrix.items():
                if skills:
                    for skill in skills:
                        # Extract frequency from skill string (e.g., "Express.js, frequency: 10")
                        if 'frequency:' in skill:
                            parts = skill.split(', frequency:')
                            if len(parts) == 2:
                                skill_name = parts[0].strip()
                                freq = int(parts[1].strip())
                                skill_freq_pairs.append((skill_name, freq))
            
            # Sort by frequency and display top 5
            sorted_skills = sorted(skill_freq_pairs, key=lambda x: x[1], reverse=True)
            for skill, freq in sorted_skills[:5]:
                print(f"  • {skill}: {freq} PRs")
        
        # Show JavaScript expertise matrix
        if 'javascript_skill_matrix' in profile:
            print("\nJavaScript Expertise Areas:")
            js_matrix = profile['javascript_skill_matrix']
            for category, skills in js_matrix.items():
                if skills:
                    # Clean category name (remove frequency info)
                    clean_category = category.split(', frequency:')[0]
                    print(f"  {clean_category}:")
                    for skill in skills[:3]:  # Top 3 in each category
                        # Clean skill name (remove frequency info)
                        clean_skill = skill.split(', frequency:')[0]
                        freq_info = skill.split(', frequency:')[1] if ', frequency:' in skill else '0'
                        print(f"    - {clean_skill} ({freq_info}x)")
        
        print(f"\nSummary: {profile.get('summary', 'No summary available')}")
        print("=" * 50)
    
    def _extract_code_changes(self, pr_data: Dict, max_chars: int = 4000) -> str:
        """Extract compact code-change context from common PR JSON fields."""
        candidate_keys = [
            'code_changes', 'patch', 'diff', 'files', 'file_changes',
            'changed_files', 'changed_files_data'
        ]

        chunks = []
        for key in candidate_keys:
            value = pr_data.get(key)
            if not value:
                continue

            if isinstance(value, str):
                chunks.append(value)
            elif isinstance(value, list):
                for item in value:
                    if isinstance(item, str):
                        chunks.append(item)
                    elif isinstance(item, dict):
                        filename = item.get('filename') or item.get('file') or item.get('path', 'unknown_file')
                        patch = item.get('patch') or item.get('diff') or item.get('changes') or item.get('content')
                        if patch:
                            chunks.append(f"File: {filename}\n{patch}")
            elif isinstance(value, dict):
                chunks.append(json.dumps(value, ensure_ascii=False))

        if not chunks:
            return "No code change details available in this PR payload."

        combined = "\n\n".join(chunks)
        return combined[:max_chars]

    def _safe_parse_json_object(self, text: str) -> Dict:
        """Parse first valid JSON object from LLM output, tolerating markdown wrappers."""
        import re

        cleaned = (text or "").strip()
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
        cleaned = re.sub(r"\s*```$", "", cleaned)

        decoder = json.JSONDecoder()

        # Try parsing from every '{' position until one succeeds.
        for idx, ch in enumerate(cleaned):
            if ch != '{':
                continue
            try:
                parsed, _ = decoder.raw_decode(cleaned[idx:])
                if isinstance(parsed, dict):
                    return parsed
            except json.JSONDecodeError:
                continue

        # Last attempt: greedily extract object-like content.
        match = re.search(r"\{[\s\S]*\}", cleaned)
        if match:
            parsed = json.loads(match.group(0))
            if isinstance(parsed, dict):
                return parsed

        raise ValueError("No valid JSON object found in model output")

    def analyze_pr_requirements(self, pr_data: Dict) -> Dict:
        """Analyze what technical skills and expertise this PR requires for review"""
        
        # Get PR summary + code changes
        pr_content = reviewer_loader.get_pr_summary(pr_data)
        code_changes = self._extract_code_changes(pr_data)
        
        try:
            # Analyze PR requirements
            analysis_result = self.analysis_chain.run(
                pr_content=pr_content,
                code_changes=code_changes
            )
            requirements = self._safe_parse_json_object(analysis_result)
            return requirements
                
        except Exception as e:
            print(f"❌ Error analyzing PR requirements: {e}")
            return self._fallback_analysis(pr_data)
    
    def _fallback_analysis(self, pr_data: Dict) -> Dict:
        """Fallback analysis based on PR metadata"""
        labels = pr_data.get('labels', [])
        title = pr_data.get('title', '').lower()
        description = pr_data.get('description', '').lower()
        
        # Determine primary language (default to JavaScript for Express repo)
        primary_lang = "JavaScript"
        
        # Determine expertise areas based on content
        expertise_areas = ["General Development"]
        if any(term in title + description for term in ['test', 'spec']):
            expertise_areas.append("Testing")
        if any(term in title + description for term in ['doc', 'readme']):
            expertise_areas.append("Documentation")
        if any(term in title + description for term in ['security', 'vulnerability']):
            expertise_areas.append("Security")
        if any(term in title + description for term in ['performance', 'optimization']):
            expertise_areas.append("Performance")
        
        return {
            "technical_skills_needed": [primary_lang, "Node.js", "Express.js"],
            "review_expertise_areas_needed": expertise_areas,
            "complexity_level": "Medium",
            "review_focus_areas": ["Code Quality", "Functionality"],
            "primary_language": primary_lang,
            "frameworks_involved": ["Express.js", "Node.js"],
            "review_type_needed": "Code Quality"
        }
    
    def get_similar_reviews(self, pr_requirements: Dict, k: int = 3) -> str:
        """Get similar past reviews to help with reviewer matching"""
        try:
            # Create query from requirements
            query_parts = []
            query_parts.extend(pr_requirements.get('technical_skills_needed', []))
            query_parts.extend(pr_requirements.get('review_expertise_areas_needed', []))
            query_parts.append(pr_requirements.get('primary_language', ''))
            
            query = ' '.join(query_parts)
            
            # Find similar reviewed PRs
            similar_docs = self.vector_store.find_similar_reviewed_prs(query, k=k)
            
            similar_reviews_summary = ""
            for i, doc in enumerate(similar_docs, 1):
                metadata = doc.metadata
                reviewers = ', '.join(metadata.get('reviewers', []))
                similar_reviews_summary += f"""
SIMILAR REVIEW {i}:
PR #{metadata.get('pr_number')}: {metadata.get('title', 'No title')}
Reviewers: {reviewers}
Review Count: {metadata.get('review_count', 0)}
Content: {doc.page_content[:300]}...

"""
            
            return similar_reviews_summary
            
        except Exception as e:
            print(f"❌ Error getting similar reviews: {e}")
            return "No similar reviews found"
    
    def find_best_reviewers(self, pr_requirements: Dict, top_k: int = 3) -> Dict:
        """Find the best reviewers for this PR"""
        
        # Format reviewer profiles for the LLM with frequency-enhanced data
        profiles_summary = ""
        for reviewer_name, profile_data in self.reviewer_profiles.items():
            stats = profile_data.get('stats', {})
            profile = profile_data.get('profile', {})
            
            # Extract top skills from JavaScript skill matrix if available
            top_skills = []
            if 'javascript_skill_matrix' in profile:
                js_matrix = profile['javascript_skill_matrix']
                skill_freq_pairs = []
                
                # Extract skills and frequencies from the matrix
                for category, skills in js_matrix.items():
                    if skills:
                        for skill in skills:
                            if 'frequency:' in skill:
                                parts = skill.split(', frequency:')
                                if len(parts) == 2:
                                    skill_name = parts[0].strip()
                                    freq = int(parts[1].strip())
                                    skill_freq_pairs.append((skill_name, freq))
                
                # Sort by frequency and get top 5
                sorted_skills = sorted(skill_freq_pairs, key=lambda x: x[1], reverse=True)
                top_skills = [f"{skill} ({freq}x)" for skill, freq in sorted_skills[:5]]
            
            # Fallback to primary skills if no frequency data
            if not top_skills:
                top_skills = profile.get('primary_skills', [])
            
            # Get JavaScript-specific expertise if available
            js_expertise = ""
            if 'javascript_skill_matrix' in profile:
                js_matrix = profile['javascript_skill_matrix']
                # Look for framework expertise (clean the category names)
                framework_expertise = []
                backend_skills = []
                
                for category, skills in js_matrix.items():
                    clean_category = category.split(', frequency:')[0].lower()
                    if 'framework' in clean_category or 'library' in clean_category:
                        framework_expertise = [skill.split(', frequency:')[0] for skill in skills]
                    elif 'backend' in clean_category:
                        backend_skills = [skill.split(', frequency:')[0] for skill in skills]
                
                if framework_expertise or backend_skills:
                    js_expertise = f"\n- JavaScript Expertise: {', '.join(framework_expertise[:3] + backend_skills[:3])}"
            
            # Get a sample of review details
            review_details = profile_data.get('stats', {}).get('review_details', [])
            
            sample_reviews = ""
            for review in review_details[:3]:  # First 3 reviews as examples
                sample_reviews += f"\n    - PR #{review.get('pr_number')}: {review.get('review_state')} - {review.get('review_body', 'No comment')[:100]}..."
            
            profiles_summary += f"""
REVIEWER: {reviewer_name}
- Experience Level: {profile.get('experience_level', 'Unknown')}
- Total Reviews: {stats.get('total_reviews', 0)}
- Total Comments: {stats.get('total_comments', 0)}
- Total PRs Reviewed: {stats.get('total_prs', 0)}
- Programming Languages: {', '.join(profile.get('programming_languages', []))}
- Top Skills (by frequency): {', '.join(top_skills)}
- Primary Skills: {', '.join(profile.get('primary_skills', []))}{js_expertise}
- Repositories: {', '.join(stats.get('repos', []))}
- Summary: {profile.get('summary', 'No summary available')[:200]}...
- Sample Review Patterns:{sample_reviews}

"""
        
        # Get similar past reviews
        similar_reviews = self.get_similar_reviews(pr_requirements)
        
        try:
            # Get matching recommendations
            matching_result = self.matching_chain.run(
                pr_requirements=json.dumps(pr_requirements, indent=2),
                reviewer_profiles=profiles_summary,
                similar_reviews=similar_reviews
            )
            recommendations = self._safe_parse_json_object(matching_result)
            return recommendations
                
        except Exception as e:
            print(f"❌ Error finding best reviewers: {e}")
            return self._fallback_matching(pr_requirements)
    
    def _fallback_matching(self, pr_requirements: Dict) -> Dict:
        """Enhanced fallback matching using skill frequency data from JavaScript matrix"""
        
        primary_lang = pr_requirements.get('primary_language', 'JavaScript')
        needed_skills = pr_requirements.get('technical_skills_needed', [])
        needed_frameworks = pr_requirements.get('frameworks_involved', [])
        
        # Enhanced scoring based on skill frequency, expertise, and review activity
        reviewer_scores = []
        for reviewer_name, profile_data in self.reviewer_profiles.items():
            score = 0.0
            stats = profile_data.get('stats', {})
            profile = profile_data.get('profile', {})
            
            # Get all available skills
            reviewer_skills = profile.get('primary_skills', []) + profile.get('programming_languages', [])
            
            # Extract skill frequencies from JavaScript skill matrix
            skill_frequencies = {}
            if 'javascript_skill_matrix' in profile:
                js_matrix = profile['javascript_skill_matrix']
                for category, skills in js_matrix.items():
                    if skills:
                        for skill in skills:
                            if 'frequency:' in skill:
                                parts = skill.split(', frequency:')
                                if len(parts) == 2:
                                    skill_name = parts[0].strip()
                                    freq = int(parts[1].strip())
                                    skill_frequencies[skill_name] = freq
            
            # Language match with frequency bonus
            if primary_lang in profile.get('programming_languages', []):
                base_score = 0.3
                # Bonus for high frequency of that language
                if primary_lang in skill_frequencies:
                    frequency_bonus = min(skill_frequencies[primary_lang] / 15, 0.2)  # Max 0.2 bonus
                    score += base_score + frequency_bonus
                else:
                    score += base_score
            
            # Skill overlap with frequency weighting
            for skill in needed_skills:
                if skill in skill_frequencies:
                    # Weight by skill frequency (more experience = higher score)
                    frequency_weight = min(skill_frequencies[skill] / 8, 0.25)  # Max 0.25 per skill
                    score += frequency_weight
                elif skill in reviewer_skills:
                    score += 0.08  # Basic skill match
            
            # Framework expertise bonus
            if 'javascript_skill_matrix' in profile:
                js_matrix = profile['javascript_skill_matrix']
                # Look for framework expertise in any category containing "framework" or "library"
                for category, skills in js_matrix.items():
                    clean_category = category.split(', frequency:')[0].lower()
                    if 'framework' in clean_category or 'library' in clean_category:
                        framework_expertise = [skill.split(', frequency:')[0] for skill in skills]
                        for framework in needed_frameworks:
                            if any(framework.lower() in expertise.lower() for expertise in framework_expertise):
                                score += 0.15
            
            # Experience level bonus
            experience_level = profile.get('experience_level', 'Junior')
            complexity = pr_requirements.get('complexity_level', 'Medium')
            if (complexity == 'High' and experience_level in ['Senior', 'Expert']) or \
               (complexity == 'Medium' and experience_level in ['Mid', 'Senior', 'Expert']) or \
               (complexity == 'Low'):
                score += 0.1
            
            # Review activity scoring (reduced weight to balance with skill scoring)
            total_reviews = stats.get('total_reviews', 0)
            if total_reviews > 0:
                score += min(total_reviews / 100, 0.2)  # Max 0.2 for review count
            
            # Score based on PR count
            total_prs = stats.get('total_prs', 0)
            if total_prs > 0:
                score += min(total_prs / 60, 0.15)  # Max 0.15 for PR count
            
            # Score based on comment engagement
            total_comments = stats.get('total_comments', 0)
            if total_comments > 0:
                score += min(total_comments / 40, 0.1)  # Max 0.1 for comments
            
            reviewer_scores.append((reviewer_name, score))
        
        # Sort by score and take top 3
        reviewer_scores.sort(key=lambda x: x[1], reverse=True)
        top_3 = reviewer_scores[:3]
        
        recommendations = []
        for i, (reviewer_name, score) in enumerate(top_3):
            stats = self.reviewer_profiles[reviewer_name].get('stats', {})
            profile = self.reviewer_profiles[reviewer_name].get('profile', {})
            
            # Get top skills for this reviewer
            top_skills = []
            if 'javascript_skill_matrix' in profile:
                js_matrix = profile['javascript_skill_matrix']
                skill_freq_pairs = []
                
                # Extract skills and frequencies from the matrix
                for category, skills in js_matrix.items():
                    if skills:
                        for skill in skills:
                            if 'frequency:' in skill:
                                parts = skill.split(', frequency:')
                                if len(parts) == 2:
                                    skill_name = parts[0].strip()
                                    freq = int(parts[1].strip())
                                    skill_freq_pairs.append((skill_name, freq))
                
                # Sort by frequency and get top 3
                sorted_skills = sorted(skill_freq_pairs, key=lambda x: x[1], reverse=True)
                top_skills = [skill for skill, freq in sorted_skills[:3]]
            
            # Fallback to primary skills
            if not top_skills:
                top_skills = profile.get('primary_skills', [])[:3]
            
            # Generate more detailed reasoning
            reasoning_parts = []
            if primary_lang in profile.get('programming_languages', []):
                reasoning_parts.append(f"Strong {primary_lang} experience")
            
            # Check for skill matches in the extracted frequencies
            skill_matches = []
            if 'javascript_skill_matrix' in profile:
                js_matrix = profile['javascript_skill_matrix']
                for category, skills in js_matrix.items():
                    for skill in skills:
                        skill_name = skill.split(', frequency:')[0].strip()
                        if skill_name in needed_skills:
                            skill_matches.append(skill_name)
            
            if skill_matches:
                reasoning_parts.append(f"Experienced in {', '.join(skill_matches[:2])}")
            
            reasoning_parts.append(f"Active reviewer with {stats.get('total_reviews', 0)} reviews")
            
            if not reasoning_parts:
                reasoning_parts.append("General review skills match")
            
            recommendations.append({
                "reviewer_name": reviewer_name,
                "match_score": min(score, 1.0),
                "reasoning": "; ".join(reasoning_parts),
                "strengths_alignment": top_skills,
                "review_experience": f"Has reviewed {stats.get('total_prs', 0)} PRs with {stats.get('total_comments', 0)} comments",
                "potential_concerns": "Limited specific expertise" if score < 0.5 else "None identified"
            })
        
        confidence = "High" if top_3[0][1] > 0.7 else ("Medium" if top_3[0][1] > 0.4 else "Low")
        
        return {
            "recommended_reviewers": recommendations,
            "assignment_confidence": confidence,
            "assignment_reasoning": "Enhanced matching using skill frequency and review activity data"
        }
    
    def smart_reviewer_assignment(self, pr_data: Dict) -> Tuple[str, Dict]:
        """Complete workflow: analyze PR, find best reviewers"""
        
        print(f"🔍 Analyzing PR #{pr_data.get('pr_number')}: {pr_data.get('title')}")
        print("=" * 80)
        
        # Step 1: Analyze PR requirements
        print("📊 Step 1: Analyzing PR review requirements...")
        pr_requirements = self.analyze_pr_requirements(pr_data)
        
        print(f"✅ Required Skills: {', '.join(pr_requirements.get('technical_skills_needed', []))}")
        print(f"✅ Review Areas: {', '.join(pr_requirements.get('review_expertise_areas_needed', []))}")
        print(f"✅ Complexity: {pr_requirements.get('complexity_level', 'Unknown')}")
        print(f"✅ Review Type: {pr_requirements.get('review_type_needed', 'Unknown')}")
        
        # Step 2: Find best reviewers
        print("\n👥 Step 2: Finding best-matched reviewers...")
        reviewer_recommendations = self.find_best_reviewers(pr_requirements)
        
        print("🎯 Recommended Reviewers:")
        for i, rec in enumerate(reviewer_recommendations.get('recommended_reviewers', []), 1):
            print(f"  {i}. {rec['reviewer_name']} (Score: {rec['match_score']:.2f})")
            print(f"     Reasoning: {rec['reasoning']}")
            print(f"     Experience: {rec.get('review_experience', 'Not specified')}")
            print(f"     Strengths: {', '.join(rec.get('strengths_alignment', []))}")
            if rec.get('potential_concerns'):
                print(f"     Concerns: {rec['potential_concerns']}")
            print()
        
        # Combine results
        assignment_summary = f"""
🎯 SMART REVIEWER ASSIGNMENT SUMMARY
{'=' * 50}

📋 PR REVIEW REQUIREMENTS:
- Technical Skills Needed: {', '.join(pr_requirements.get('technical_skills_needed', []))}
- Review Expertise Areas: {', '.join(pr_requirements.get('review_expertise_areas_needed', []))}
- Complexity Level: {pr_requirements.get('complexity_level', 'Unknown')}
- Primary Language: {pr_requirements.get('primary_language', 'Unknown')}
- Review Type Needed: {pr_requirements.get('review_type_needed', 'Unknown')}

👥 RECOMMENDED REVIEWERS:
"""
        
        for i, rec in enumerate(reviewer_recommendations.get('recommended_reviewers', []), 1):
            assignment_summary += f"""
{i}. **{rec['reviewer_name']}** (Match Score: {rec['match_score']:.2f})
   - Reasoning: {rec['reasoning']}
   - Review Experience: {rec.get('review_experience', 'Not specified')}
   - Key Strengths: {', '.join(rec.get('strengths_alignment', [])[:3])}
   - Concerns: {rec.get('potential_concerns', 'None identified')}
"""
        
        assignment_summary += f"""
🔍 Assignment Confidence: {reviewer_recommendations.get('assignment_confidence', 'Unknown')}
📝 Assignment Reasoning: {reviewer_recommendations.get('assignment_reasoning', 'Not provided')}

{'=' * 50}
"""
        
        return assignment_summary, {
            'pr_requirements': pr_requirements,
            'reviewer_recommendations': reviewer_recommendations
        }

# Create the smart reviewer assigner
print("🚀 Creating Smart Reviewer Assigner...")
smart_reviewer_assigner = SmartReviewerAssigner(llm, reviewer_vector_store, "../Reviewers Profiles")
print("✅ Smart Reviewer Assigner ready!")

🚀 Creating Smart Reviewer Assigner...
🔍 Loading reviewer profiles from ../Reviewers Profiles/...
✅ Loaded 24 reviewer profiles
👥 Available reviewers: abhisekp, aravindvnair99, bjohansebas, CBID2, ctcpip, dpopp07, hamidrezaghavami, Hardanish-Singh, IamLizu, inigomarquinez, jonchurch, kjugi, krzysdz, lukaselmer, mertcanaltin, Phillip9587, RaginiSharma01, rileyjshaw, RobinTail, sheplu, shivarm, tausiq2003, UlisesGascon, wesleytodd
✅ Smart Reviewer Assigner ready!


In [29]:
# Quick Reviewer Assignment Function - for easy testing with different PRs
def quick_assign_reviewer(pr_number: int):
    """Quick function to assign reviewers to a PR by number"""
    # Find PR by number
    target_pr = None
    for pr in reviewer_loader.reviewer_data:
        if pr.get('pr_number') == pr_number:
            target_pr = pr
            break
    
    if not target_pr:
        print(f"❌ PR #{pr_number} not found in loaded reviewer data")
        return
    
    print(f"🎯 SMART REVIEWER ASSIGNMENT FOR PR #{pr_number}")
    print(f"📝 Title: {target_pr.get('title')}")
    print(f"👤 Author: {target_pr.get('author', {}).get('username', 'Unknown')}")
    print("=" * 60)
    
    # Get assignment recommendations
    pr_requirements = smart_reviewer_assigner.analyze_pr_requirements(target_pr)
    reviewer_recommendations = smart_reviewer_assigner.find_best_reviewers(pr_requirements)
    
    # Get the best reviewer (first in the list)
    recommended_reviewers = reviewer_recommendations.get('recommended_reviewers', [])
    if not recommended_reviewers:
        print("❌ No suitable reviewer found for this PR")
        return
    
    best_reviewer = recommended_reviewers[0]  # Get only the top recommendation
    
    print("🎯 ASSIGNED REVIEWER:")
    print(f"\n👨 **{best_reviewer['reviewer_name']}** (Match: {best_reviewer['match_score']:.0%})")
    print(f"💡 Why: {best_reviewer['reasoning']}")
    print(f"⭐ Strengths: {', '.join(best_reviewer.get('strengths_alignment', [])[:3])}")
    if best_reviewer.get('potential_concerns'):
        print(f"⚠️ Concerns: {best_reviewer['potential_concerns']}")
    
    print(f"\n🔍 Assignment Confidence: {reviewer_recommendations.get('assignment_confidence', 'Unknown')}")
    print("=" * 60)


quick_assign_reviewer(6117)

🎯 SMART REVIEWER ASSIGNMENT FOR PR #6117
📝 Title: deps: upgrade ejs
👤 Author: agungjati


c:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


🎯 ASSIGNED REVIEWER:

👨 **bjohansebas** (Match: 92%)
💡 Why: Strongest frequency evidence for Node.js/Express.js (6x, 5x) and dependency management (5x npm/yarn). Reviewed 64 PRs with 32 comments, showing consistent expertise in dependency compatibility and backend JavaScript. Directly matches all required skills and review focus areas.
⭐ Strengths: 6x Node.js, 5x Express.js, 5x npm/yarn (dependency management), Reviewed 64 PRs with 32 comments (high activity), Primary skills include Dependency Management and CI/CD (relevant to compatibility)
⚠️ Concerns: None identified

🔍 Assignment Confidence: High


In [30]:
# Bulk Reviewer Assignment Function - Assigns all PRs to reviewers
def assign_all_prs_to_reviewers(max_prs: int = None, include_detailed_analysis: bool = False):
    """
    Assign all PRs to the best single reviewer and return results in JSON format
    
    Args:
        max_prs: Maximum number of PRs to process (None for all)
        include_detailed_analysis: Whether to include detailed PR analysis in output
    
    Returns:
        dict: JSON-formatted results with PR assignments
    """
    print("🚀 BULK REVIEWER ASSIGNMENT STARTING...")
    print("=" * 60)
    
    if not reviewer_loader.reviewer_data:
        return {"error": "No reviewer data available"}
    
    # Determine how many PRs to process
    prs_to_process = reviewer_loader.reviewer_data
    if max_prs:
        prs_to_process = prs_to_process[:max_prs]
    
    print(f"📊 Processing {len(prs_to_process)} PRs for reviewer assignment...")
    
    # Results container
    assignment_results = {
        "metadata": {
            "total_prs_processed": len(prs_to_process),
            "processing_date": "2025-10-21",
            "available_reviewers": list(smart_reviewer_assigner.reviewer_profiles.keys()),
            "total_available_reviewers": len(smart_reviewer_assigner.reviewer_profiles)
        },
        "pr_assignments": []
    }
    
    # Process each PR
    for i, pr in enumerate(prs_to_process, 1):
        pr_number = pr.get('pr_number')
        print(f"🔄 Processing PR #{pr_number} ({i}/{len(prs_to_process)})...")
        
        try:
            # Analyze PR requirements
            pr_requirements = smart_reviewer_assigner.analyze_pr_requirements(pr)
            reviewer_recommendations = smart_reviewer_assigner.find_best_reviewers(pr_requirements, top_k=3)
            
            # Get the top 3 reviewers
            recommended_reviewers_list = reviewer_recommendations.get('recommended_reviewers', [])
            
            if recommended_reviewers_list:
                # Extract top 3 reviewers with their details
                top_3_reviewers = []
                for reviewer_data in recommended_reviewers_list[:3]:
                    top_3_reviewers.append({
                        "reviewer_name": reviewer_data['reviewer_name'],
                        "match_score": round(reviewer_data['match_score'], 2),
                        "reasoning": reviewer_data['reasoning'],
                        "strengths_alignment": reviewer_data.get('strengths_alignment', []),
                        "review_experience": reviewer_data.get('review_experience', 'Not specified'),
                        "potential_concerns": reviewer_data.get('potential_concerns', 'None identified')
                    })
                
                # Primary reviewer (best match)
                primary_reviewer = recommended_reviewers_list[0]
                assigned_reviewer = primary_reviewer['reviewer_name']
                match_score = primary_reviewer['match_score']
                reasoning = primary_reviewer['reasoning']
                strengths = primary_reviewer.get('strengths_alignment', [])
                concerns = primary_reviewer.get('potential_concerns', 'None identified')
                review_experience = primary_reviewer.get('review_experience', 'Not specified')
                confidence = reviewer_recommendations.get('assignment_confidence', 'Unknown')
            else:
                top_3_reviewers = []
                assigned_reviewer = None
                match_score = 0.0
                reasoning = "No suitable reviewer found"
                strengths = []
                concerns = "No reviewer could be matched to this PR"
                review_experience = "N/A"
                confidence = "Low"
            
            # Create PR assignment record with TOP 3 REVIEWERS
            pr_assignment = {
                "pr_number": pr_number,
                "pr_title": pr.get('title', 'No title'),
                "pr_author": pr.get('author', {}).get('username', 'Unknown') if pr.get('author') else 'Unknown',
                "pr_state": pr.get('state', 'unknown'),
                "assigned_reviewer": assigned_reviewer,  # Primary/best reviewer
                "top_3_reviewers": top_3_reviewers,  # NEW: Top 3 ranked reviewers
                "match_score": round(match_score, 2),
                "assignment_reasoning": reasoning,
                "assignment_confidence": confidence,
                "current_reviews_count": len(pr.get('reviews', [])),
                "pr_url": f"https://github.com/expressjs/express/pull/{pr_number}",
                "reviewer_strengths": strengths[:3],  # Top 3 strengths
                "potential_concerns": concerns,
                "review_experience": review_experience
            }
            
            # Add detailed analysis if requested
            if include_detailed_analysis:
                pr_assignment["detailed_analysis"] = {
                    "technical_skills_needed": pr_requirements.get('technical_skills_needed', []),
                    "review_expertise_areas": pr_requirements.get('review_expertise_areas_needed', []),
                    "complexity_level": pr_requirements.get('complexity_level', 'Unknown'),
                    "primary_language": pr_requirements.get('primary_language', 'Unknown'),
                    "review_type_needed": pr_requirements.get('review_type_needed', 'Unknown'),
                    "assignment_details": {
                        "full_reasoning": reasoning,
                        "strengths_alignment": strengths,
                        "review_experience_details": review_experience,
                        "potential_limitations": concerns
                    }
                }
            
            assignment_results["pr_assignments"].append(pr_assignment)
            
        except Exception as e:
            print(f"❌ Error processing PR #{pr_number}: {e}")
            # Add error record
            assignment_results["pr_assignments"].append({
                "pr_number": pr_number,
                "pr_title": pr.get('title', 'No title'),
                "pr_author": pr.get('author', {}).get('username', 'Unknown') if pr.get('author') else 'Unknown',
                "assigned_reviewer": None,
                "match_score": 0.0,
                "assignment_reasoning": f"Error during assignment: {str(e)}",
                "assignment_confidence": "Error",
                "error": True
            })
    
    # Generate summary statistics
    successful_assignments = len([a for a in assignment_results["pr_assignments"] if a.get('assigned_reviewer')])
    failed_assignments = len(assignment_results["pr_assignments"]) - successful_assignments
    
    # Count assignments per reviewer
    reviewer_assignment_counts = {}
    for assignment in assignment_results["pr_assignments"]:
        reviewer = assignment.get('assigned_reviewer')
        if reviewer:
            reviewer_assignment_counts[reviewer] = reviewer_assignment_counts.get(reviewer, 0) + 1
    
    assignment_results["summary"] = {
        "successful_assignments": successful_assignments,
        "failed_assignments": failed_assignments,
        "success_rate": round((successful_assignments / len(assignment_results["pr_assignments"])) * 100, 1),
        "reviewer_workload_distribution": reviewer_assignment_counts,
        "most_assigned_reviewer": max(reviewer_assignment_counts.items(), key=lambda x: x[1])[0] if reviewer_assignment_counts else None,
        "average_match_score": round(
            sum([a.get('match_score', 0) for a in assignment_results["pr_assignments"] if a.get('assigned_reviewer')]) / 
            max(successful_assignments, 1), 2
        )
    }
    
    print(f"✅ BULK ASSIGNMENT COMPLETED!")
    print(f"📊 Successfully assigned: {successful_assignments}/{len(assignment_results['pr_assignments'])} PRs")
    print(f"🎯 Success rate: {assignment_results['summary']['success_rate']}%")
    print(f"⭐ Average match score: {assignment_results['summary']['average_match_score']}")
    print("=" * 60)
    
    return assignment_results

def save_assignments_to_file(assignments_data: dict, filename: str = "pr_reviewer_assignments.json"):
    """Save the assignment results to a JSON file"""
    try:
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(assignments_data, f, indent=2, ensure_ascii=False)
        print(f"✅ Assignments saved to {filename}")
        return True
    except Exception as e:
        print(f"❌ Error saving to file: {e}")
        return False

def display_assignment_summary(assignments_data: dict):
    """Display a nice summary of the assignments"""
    print("📋 REVIEWER ASSIGNMENT SUMMARY")
    print("=" * 50)
    
    summary = assignments_data.get('summary', {})
    metadata = assignments_data.get('metadata', {})
    
    print(f"📊 Total PRs Processed: {metadata.get('total_prs_processed', 0)}")
    print(f"✅ Successful Assignments: {summary.get('successful_assignments', 0)}")
    print(f"❌ Failed Assignments: {summary.get('failed_assignments', 0)}")
    print(f"🎯 Success Rate: {summary.get('success_rate', 0)}%")
    print(f"⭐ Average Match Score: {summary.get('average_match_score', 0)}")
    
    print(f"\n👥 REVIEWER WORKLOAD DISTRIBUTION:")
    workload = summary.get('reviewer_workload_distribution', {})
    for reviewer, count in sorted(workload.items(), key=lambda x: x[1], reverse=True):
        print(f"   {reviewer}: {count} PRs")
    
    print(f"\n🏆 Most Assigned Reviewer: {summary.get('most_assigned_reviewer', 'None')}")
    print("=" * 50)

# Example usage functions
print("🔧 BULK ASSIGNMENT FUNCTIONS READY!")
print("💡 Usage examples:")
print("   assignments = assign_all_prs_to_reviewers(max_prs=10)  # Process first 10 PRs")
print("   assignments = assign_all_prs_to_reviewers()  # Process all PRs")
print("   save_assignments_to_file(assignments, 'my_assignments.json')")
print("   display_assignment_summary(assignments)")

🔧 BULK ASSIGNMENT FUNCTIONS READY!
💡 Usage examples:
   assignments = assign_all_prs_to_reviewers(max_prs=10)  # Process first 10 PRs
   assignments = assign_all_prs_to_reviewers()  # Process all PRs
   save_assignments_to_file(assignments, 'my_assignments.json')
   display_assignment_summary(assignments)


In [31]:
# 🚀 FULL PR ASSIGNMENT - Process ALL PRs and Save to File (Single Reviewer Format)

print("🎯 PROCESSING ALL PRs FOR SINGLE REVIEWER ASSIGNMENT...")
print("=" * 60)

# Process ALL PRs with detailed analysis
all_assignments = assign_all_prs_to_reviewers(
    max_prs=None,  # Process ALL PRs
    include_detailed_analysis=True  # Include full analysis
)

# Display comprehensive summary
display_assignment_summary(all_assignments)

# Save to JSON file
filename = "all_pr_reviewer_assignments.json"
save_success = save_assignments_to_file(all_assignments, filename)

if save_success:
    print(f"\n✅ SUCCESS! All PR assignments saved to '{filename}'")
    print(f"📊 Total PRs processed: {all_assignments['metadata']['total_prs_processed']}")
    print(f"💾 File contains complete assignment data with single reviewer per PR")
else:
    print(f"\n❌ Failed to save assignments to file")

# Show breakdown by reviewer
print(f"\n📈 DETAILED REVIEWER BREAKDOWN:")
print("=" * 40)
workload = all_assignments.get('summary', {}).get('reviewer_workload_distribution', {})
total_assigned = sum(workload.values())

for reviewer, count in sorted(workload.items(), key=lambda x: x[1], reverse=True):
    percentage = (count / total_assigned * 100) if total_assigned > 0 else 0
    print(f"   {reviewer:15} → {count:2d} PRs ({percentage:4.1f}%)")

print(f"\n🎯 ASSIGNMENT EFFICIENCY:")
summary = all_assignments.get('summary', {})
print(f"   Success Rate: {summary.get('success_rate', 0)}%")
print(f"   Avg Match Score: {summary.get('average_match_score', 0)}")
print(f"   Most Active Reviewer: {summary.get('most_assigned_reviewer', 'None')}")

# Show sample of high-confidence assignments with detailed format
print(f"\n🏆 TOP ASSIGNMENTS (High Confidence):")
print("=" * 50)
high_confidence_assignments = [
    assignment for assignment in all_assignments.get('pr_assignments', [])
    if assignment.get('assignment_confidence') == 'High' and assignment.get('assigned_reviewer')
]

for i, assignment in enumerate(high_confidence_assignments[:3], 1):  # Show top 3
    pr_num = assignment.get('pr_number')
    title = assignment.get('pr_title', 'No title')[:50] + "..." if len(assignment.get('pr_title', '')) > 50 else assignment.get('pr_title', 'No title')
    reviewer = assignment.get('assigned_reviewer')
    score = assignment.get('match_score', 0)
    reasoning = assignment.get('assignment_reasoning', '')[:100] + "..."
    strengths = ', '.join(assignment.get('reviewer_strengths', [])[:3])
    concerns = assignment.get('potential_concerns', 'None identified')
    
    print(f"{i}. PR #{pr_num}: {title}")
    print(f"   👨‍💻 ASSIGNED REVIEWER: **{reviewer}** (Match: {score*100:.0f}%)")
    print(f"   💡 Why: {reasoning}")
    print(f"   ⭐ Strengths: {strengths}")
    if concerns != 'None identified' and concerns != 'Not specified':
        print(f"   ⚠️ Concerns: {concerns[:100]}...")
    print(f"   🔍 Confidence: {assignment.get('assignment_confidence', 'Unknown')}")
    print()

print("=" * 60)
print(f"✅ COMPLETE! Single reviewer per PR format saved to '{filename}'")
print("💡 Each PR now has exactly ONE assigned reviewer with detailed reasoning")
print("=" * 60)

🎯 PROCESSING ALL PRs FOR SINGLE REVIEWER ASSIGNMENT...
🚀 BULK REVIEWER ASSIGNMENT STARTING...
📊 Processing 52 PRs for reviewer assignment...
🔄 Processing PR #6534 (1/52)...


c:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


🔄 Processing PR #6535 (2/52)...


c:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


🔄 Processing PR #6408 (3/52)...


...

🔄 Processing PR #6774 (51/52)...


c:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


🔄 Processing PR #6031 (52/52)...


c:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


✅ BULK ASSIGNMENT COMPLETED!
📊 Successfully assigned: 51/52 PRs
🎯 Success rate: 98.1%
⭐ Average match score: 0.91
📋 REVIEWER ASSIGNMENT SUMMARY
📊 Total PRs Processed: 52
✅ Successful Assignments: 51
❌ Failed Assignments: 1
🎯 Success Rate: 98.1%
⭐ Average Match Score: 0.91

👥 REVIEWER WORKLOAD DISTRIBUTION:
   bjohansebas: 26 PRs
   wesleytodd: 15 PRs
   UlisesGascon: 6 PRs
   Phillip9587: 4 PRs

🏆 Most Assigned Reviewer: bjohansebas
✅ Assignments saved to all_pr_reviewer_assignments.json

✅ SUCCESS! All PR assignments saved to 'all_pr_reviewer_assignments.json'
📊 Total PRs processed: 52
💾 File contains complete assignment data with single reviewer per PR

📈 DETAILED REVIEWER BREAKDOWN:
   bjohansebas     → 26 PRs (51.0%)
   wesleytodd      → 15 PRs (29.4%)
   UlisesGascon    →  6 PRs (11.8%)
   Phillip9587     →  4 PRs ( 7.8%)

🎯 ASSIGNMENT EFFICIENCY:
   Success Rate: 98.1%
   Avg Match Score: 0.91
   Most Active Reviewer: bjohansebas

🏆 TOP ASSIGNMENTS (High Confidence):
1. PR #6534: